In [4]:
import sqlite3
import pandas as pd

class ForecastsDatabase:
    """Classe para gerenciar conexão e operações no banco SQLite."""
    
    def __init__(self, db_path):
        self.db_path = db_path

    def execute_query(self, query, params=None):
        """Executa uma query no banco de dados com tratamento de erros."""
        try:
            conn = sqlite3.connect(self.db_path)
            cursor = conn.cursor()
            if params:
                cursor.execute(query, params)
            else:
                cursor.execute(query)
            conn.commit()
            conn.close()
        except sqlite3.Error as e:
            print(f"Erro ao executar query: {e}")

    def fetch_dataframe(self, query):
        """Executa uma consulta e retorna um DataFrame do Pandas."""
        try:
            conn = sqlite3.connect(self.db_path)
            df = pd.read_sql_query(query, conn)
            conn.close()
            return df
        except sqlite3.Error as e:
            print(f"Erro ao buscar dados: {e}")
            return pd.DataFrame()  # Retorna um DataFrame vazio em caso de erro

# Caminho do banco de dados
db = ForecastsDatabase("sqforecasts.db")

# Atualizando a view antes de criar o DataFrame
db.execute_query("DROP VIEW IF EXISTS vwforecastsdados")
db.execute_query("""
CREATE VIEW vwforecastsdados AS
SELECT 
    tbt.id_titulos, tbt.symbol, tbt.valorestatistico, tbt.valortipo,
    tbf.id_forecasts, tbf.data AS data_forecast,
    tfd.id_forecastsdados, tfd.data AS data_dados, tfd.valor, tfd.error
FROM tbtitulos tbt
INNER JOIN tbforecasts tbf ON tbt.id_titulos = tbf.id_titulos
INNER JOIN tbforecastsdados tfd ON tbf.id_forecasts = tfd.id_forecasts;
""")

# Criando o DataFrame
df_forecasts = db.fetch_dataframe("SELECT * FROM vwforecastsdados")

# Exibindo os primeiros registros
print(df_forecasts.head())

   id_titulos symbol valorestatistico valortipo  id_forecasts  \
0           1    DXY            media    indice             1   
1           1    DXY            media    indice             1   
2           1    DXY            media    indice             1   
3           1    DXY            media    indice             1   
4           1    DXY            media    indice             1   

         data_forecast  id_forecastsdados           data_dados    valor  error  
0  2025-05-02 00:00:00                  1  2025-04-01 00:00:00  100.725   0.00  
1  2025-05-02 00:00:00                  2  2025-05-01 00:00:00  101.280   0.38  
2  2025-05-02 00:00:00                  3  2025-06-01 00:00:00  103.450   0.46  
3  2025-05-02 00:00:00                  4  2025-07-01 00:00:00  105.310   0.50  
4  2025-05-02 00:00:00                  5  2025-08-01 00:00:00  104.200   0.53  


In [8]:
import sqlite3
import pandas as pd

# Abrindo conexão
conn = sqlite3.connect("sqforecasts.db")

# Criando DataFrame a partir da view
df = pd.read_sql_query("SELECT * FROM vwforecastsdados", conn)

# Fechando conexão
conn.close()

# Exibindo os dados
display(df.head())

,id_titulos,symbol,valorestatistico,valortipo,id_forecasts,data_forecast,id_forecastsdados,data_dados,valor,error
0,1,DXY,media,indice,1,2025-05-02 00:00:00,1,2025-04-01 00:00:00,100.725,0.00
1,1,DXY,media,indice,1,2025-05-02 00:00:00,2,2025-05-01 00:00:00,101.280,0.38
2,1,DXY,media,indice,1,2025-05-02 00:00:00,3,2025-06-01 00:00:00,103.450,0.46
3,1,DXY,media,indice,1,2025-05-02 00:00:00,4,2025-07-01 00:00:00,105.310,0.50
4,1,DXY,media,indice,1,2025-05-02 00:00:00,5,2025-08-01 00:00:00,104.200,0.53
